In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(font_scale=2.2,style='white')
import pandas as pd
import numpy as np
import scipy
import pickle as pkl
from collections import OrderedDict 


In [2]:
with open('df_unified_20211009.pkl', 'rb') as f:
    df = pkl.load(f)

In [3]:
def get_heat_over_389_participants(df):
    test_df = df[pd.to_numeric(df.quest_heat, errors='coerce').notnull()]
    test_df['quest_heat_numeric'] = pd.to_numeric(test_df['quest_heat'])
    heat_over_389_rows = test_df.where(test_df['quest_heat_numeric'] > 38.9).where(test_df['quest_heat_numeric'] < 41).dropna(how='all')
    return list(heat_over_389_rows.index)


In [4]:
heat_over_389_participants = get_heat_over_389_participants(df)

for part in heat_over_389_participants:
    df.loc[part, "quest_symptoms"] = df.loc[part, "quest_symptoms"] + ',heat over 38.9'

<ipython-input-3-f77aac9f8aab>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['quest_heat_numeric'] = pd.to_numeric(test_df['quest_heat'])


In [5]:
df[['date','first_dose_date', 'second_dose_date', 'third_dose_date']].tail()

,date,first_dose_date,second_dose_date,third_dose_date
48899,2021-09-23 00:00:00,2021-01-05 19:30:00,2021-01-28 18:00:00,2021-09-22 08:35:00
59978,2021-09-23 00:00:00,NaN,NaN,2021-09-19 12:00:00
71234,2021-09-23 00:00:00,NaN,NaN,2021-09-18 12:37:00
93102,2021-09-23 00:00:00,2021-01-04 16:00:00,2021-01-25 16:00:00,2021-09-17 11:11:00
67318,2021-09-23 00:00:00,NaN,NaN,2021-09-23 12:35:00


In [6]:
df[['days_from_q_to_first',
       'hours_from_q_to_first', 'days_from_q_to_second',
       'hours_from_q_to_second', 'days_from_q_to_third',
       'hours_from_q_to_third']].tail()

,days_from_q_to_first,hours_from_q_to_first,days_from_q_to_second,hours_from_q_to_second,days_from_q_to_third,hours_from_q_to_third
48899,260.0,6251.453333,237.0,5700.953333,0.0,22.370000
59978,NaN,NaN,NaN,NaN,3.0,91.503056
71234,NaN,NaN,NaN,NaN,4.0,116.392778
93102,261.0,6281.548056,240.0,5777.548056,5.0,142.364722
67318,NaN,NaN,NaN,NaN,-1.0,-2.948333


In [7]:
df[['quest_sleep_time', 'quest_sleep_quality',
       'quest_sport_time', 'quest_meetings', 'quest_mood', 'quest_stress',
       'quest_symptoms', 'quest_diagnosis', 'quest_other_symptoms',
       'quest_heat', 'quest_other_diagnosis']].tail()

,quest_sleep_time,quest_sleep_quality,quest_sport_time,quest_meetings,quest_mood,quest_stress,quest_symptoms,quest_diagnosis,quest_other_symptoms,quest_heat,quest_other_diagnosis
48899,6.5,0.0,0.0,6.0,-1.0,1.0,weakness,healthy,NaN,NaN,NaN
59978,7.0,1.0,45.0,5.0,1.0,-1.0,healthy,healthy,NaN,NaN,NaN
71234,5.0,1.0,180.0,3.0,1.0,-1.0,weakness,healthy,NaN,NaN,NaN
93102,7.0,1.0,30.0,10.0,1.0,-1.0,healthy,healthy,NaN,NaN,NaN
67318,4.0,0.0,10.0,11.0,0.0,0.0,healthy,healthy,NaN,NaN,NaN


In [15]:
vaccines_to_analyze = ['days_from_q_to_first', 'days_from_q_to_second', 'days_from_q_to_third']

In [16]:
vaccine_labels = ['First Vaccination', 'Second Vaccination', 'Third Vaccination']

In [17]:
symptoms_dfs = [df[(df[v]<=7)&(df[v]>=-7)][['participant_num', 'quest_symptoms', v]] for v in vaccines_to_analyze]

In [18]:
for i in range(len(vaccine_labels)):
    print("%s n= %d" % (vaccine_labels[i], symptoms_dfs[i]["participant_num"].nunique()))

First Vaccination n= 244
Second Vaccination n= 386
Third Vaccination n= 1443


In [19]:
for symptoms_df in symptoms_dfs:
    symptoms_df["quest_symptoms_processed"] = symptoms_df["quest_symptoms"].replace({'-1': '',
                                                                             'skipped_question': '',
                                                                             np.nan: ''}, regex=True)

In [20]:
# Add helper methods and objects

#First, categorize heat and fever into > 38.9 and <= 38.9
#Severe: ['chest_pain', 'dyspnea', 'heat over 38.9', 'confusion', 'chills']
 
#Mild: everything else: ['abdominal_pain', 'feel_heat', 'heat_over_37.5', 'back_or_neck_pain', 'cold', 'muscles_pain', 'weakness', 'headache',
#'dizziness', 'vomiting', 'sore_throat', 'diarrhea', 'cough', 'hand_muscles_pain', 'leg_pain', 'ear_pain', 'taste_smell', 'lymph',
# 'fast_heartbeat', 'hypertension']


symp_titles = {'healthy':'No\n Reaction',
               'chest_pain': 'Chest\n Pain',
               'dyspnea':'Dyspnea',
               'heat over 38.9':'Fever\n >38.9',
               'confusion':'Confusion',
               'chills':'Chills',
               'abdominal_pain':'Muscle\n Pain',
               'feel_heat':'Fever\n <=38.9',
               'heat over 37.5':'Fever\n <=38.9',
               'back_or_neck_pain':'Muscle\n Pain',
               'cold':'Cold',
               'muscles_pain':'Muscle\n Pain',
               'weakness':'Fatigue',
               'headache':'Headache',
               'dizziness':'Dizziness',
               'vomiting':'Vomiting',
               'sore_throat':'Sore\n Throat',
               'diarrhea':'Diarrhea',
               'cough':'Cough',
               'hand_muscles_pain':'Muscle\n Pain',
               'leg_pain':'Muscle\n Pain',
               'ear_pain':'Muscle\n Pain',
               'taste_smell':'Loss of\nTaste/Smell',
               'lymph':'Lymph',
               'fast_heartbeat':'Fast Heartbeat',
               'hypertension':'Hypertension',
               'other':'Other',
              '' : 'Skipped Questionnaire'}

possible_symptoms = symp_titles.keys()

def classify_heat(heat_str):
    heat_array = heat_str.split(':')
    if heat_array[0] != 'heat':
        return ""
    try:
        temp = float(heat_array[1])
        # One person made a typo, heat is 636.6
        if temp >= 300:
            temp = temp - 600
        # A couple people did not report 30 with their heat, e.g. reporting 6.4 instead of 36.4
        if temp <= 10:
            temp = temp + 30
            
        if temp <= 30 or temp >= 41:
            return "error with heat %.2f" % (temp)
        
        if temp > 38.9:
            return "heat over 38.9"
        if temp > 37.5:
            return "heat over 37.5"
        else:# temp >= 35:
            return "no heat"
    # If the heat could not be converted to float, assume person is healthy
    except:
        return "no heat"

def preprocess_symptom_array(arr):
    arr = arr.copy()
    for i in range(len(arr)):
        s = arr[i]
        if "heat:" in s:
            arr[i] = classify_heat(s)
    return list(set(arr))

In [21]:
# Assert that all symptoms are accounted for in the symptoms listed
# in the above dictionary or by processing their heat entry

for symptoms_df in symptoms_dfs:
    for i,r in symptoms_df.iterrows():
        symptoms = r["quest_symptoms_processed"].split(',')
        for symptom in symptoms:
            #print(symptoms)
            assert((symptom in possible_symptoms)
                   or (classify_heat(symptom) in possible_symptoms)
                   or (classify_heat(symptom) == 'no heat'))

In [22]:
# Create baseline and make sure that those who reported symptoms
# in days [-7,...,-1] don't have the same symptom reported for day 0

def create_participant_to_symptoms_before_vaccine_dict(vaccine_num):
    days_since_vaccine = vaccines_to_analyze[vaccine_num]
    symptoms_df = symptoms_dfs[vaccine_num]
    
    part_to_symptoms_before_vaccine = dict()

    df = symptoms_df.where(symptoms_df[days_since_vaccine]<=-1).dropna(how='all')
    df = df.where(df["quest_symptoms_processed"] != '').dropna(how='all').drop_duplicates('participant_num', keep='last')

    for i,r in df.iterrows():
        symptoms_old = r["quest_symptoms_processed"].split(',')
        symptoms = preprocess_symptom_array(symptoms_old)


        participant = r["participant_num"]

        if symptoms == [''] or symptoms == ['skipped_question'] or symptoms == ['-1']:
            part_to_symptoms_before_vaccine[participant] = []
        elif 'healthy' in symptoms or symptoms == ['no heat']:
            part_to_symptoms_before_vaccine[participant] = []
        else:
            actual_symptoms = list()

            for s in symptoms:
                if s == 'no heat':
                    part_to_symptoms_before_vaccine[participant] = []
                    break
                elif s in possible_symptoms and s != '':
                    actual_symptoms.append(s)

            part_to_symptoms_before_vaccine[participant] = actual_symptoms

        print("Participant %s Symptoms %s => %s" % (participant, symptoms, part_to_symptoms_before_vaccine[participant]))
    return part_to_symptoms_before_vaccine

In [23]:
part_to_sbv_dicts = [create_participant_to_symptoms_before_vaccine_dict(i) for i in range(3)]

Participant p1_152 Symptoms ['healthy'] => []
Participant p1_3 Symptoms ['headache'] => ['headache']
Participant p1_23 Symptoms ['healthy'] => []
Participant 73 Symptoms ['healthy'] => []
Participant p1_296 Symptoms ['healthy'] => []
Participant p1_226 Symptoms ['healthy'] => []
Participant p1_123 Symptoms ['healthy'] => []
Participant p1_32 Symptoms ['other'] => ['other']
Participant 7 Symptoms ['other'] => ['other']
Participant p1_31 Symptoms ['healthy'] => []
Participant p1_28 Symptoms ['healthy'] => []
Participant p1_24 Symptoms ['healthy'] => []
Participant p1_132 Symptoms ['healthy'] => []
Participant 59 Symptoms ['healthy'] => []
Participant 83 Symptoms ['sore_throat'] => ['sore_throat']
Participant 115 Symptoms ['healthy'] => []
Participant 75 Symptoms ['healthy'] => []
Participant p1_22 Symptoms ['healthy'] => []
Participant p1_21 Symptoms ['healthy'] => []
Participant p1_6 Symptoms ['healthy'] => []
Participant p1_144 Symptoms ['healthy'] => []
Participant 57 Symptoms ['healt

Participant p1_594 Symptoms ['healthy'] => []
Participant p1_592 Symptoms ['headache'] => ['headache']
Participant p1_582 Symptoms ['healthy'] => []
Participant 181 Symptoms ['healthy'] => []
Participant 162 Symptoms ['healthy'] => []
Participant p1_596 Symptoms ['cold'] => ['cold']
Participant p1_599 Symptoms ['healthy'] => []
Participant p1_595 Symptoms ['healthy'] => []
Participant 101 Symptoms ['healthy'] => []
Participant 212 Symptoms ['healthy'] => []
Participant p1_608 Symptoms ['abdominal_pain'] => ['abdominal_pain']
Participant 110 Symptoms ['healthy'] => []
Participant p1_254 Symptoms ['healthy'] => []
Participant 31 Symptoms ['healthy'] => []
Participant p1_609 Symptoms ['healthy'] => []
Participant 68 Symptoms ['cough', 'weakness'] => ['cough', 'weakness']
Participant p1_600 Symptoms ['healthy'] => []
Participant p1_604 Symptoms ['sore_throat'] => ['sore_throat']
Participant p1_601 Symptoms ['weakness'] => ['weakness']
Participant 36 Symptoms ['healthy'] => []
Participant p

Participant 2452 Symptoms ['healthy'] => []
Participant 1884 Symptoms ['healthy'] => []
Participant 1192 Symptoms ['healthy'] => []
Participant 502 Symptoms ['healthy'] => []
Participant 1113 Symptoms ['healthy'] => []
Participant 1629 Symptoms ['healthy'] => []
Participant 2409 Symptoms ['healthy'] => []
Participant 733 Symptoms ['healthy'] => []
Participant 1315 Symptoms ['healthy'] => []
Participant 1736 Symptoms ['healthy'] => []
Participant 693 Symptoms ['healthy'] => []
Participant 770 Symptoms ['abdominal_pain', 'weakness'] => ['abdominal_pain', 'weakness']
Participant 1324 Symptoms ['healthy'] => []
Participant 769 Symptoms ['healthy'] => []
Participant 2308 Symptoms ['healthy'] => []
Participant 1548 Symptoms ['healthy'] => []
Participant 297 Symptoms ['healthy'] => []
Participant 1145 Symptoms ['cold'] => ['cold']
Participant 2104 Symptoms ['healthy'] => []
Participant 862 Symptoms ['healthy'] => []
Participant 2168 Symptoms ['healthy'] => []
Participant 1114 Symptoms ['healt

Participant 871 Symptoms ['healthy'] => []
Participant 2510 Symptoms ['healthy'] => []
Participant 2423 Symptoms ['healthy'] => []
Participant 721 Symptoms ['healthy'] => []
Participant 2670 Symptoms ['chills'] => ['chills']
Participant 2816 Symptoms ['healthy'] => []
Participant 2918 Symptoms ['healthy'] => []
Participant 365 Symptoms ['healthy'] => []
Participant 1371 Symptoms ['headache', 'weakness'] => ['headache', 'weakness']
Participant 1514 Symptoms ['sore_throat'] => ['sore_throat']
Participant 2514 Symptoms ['healthy'] => []
Participant 2214 Symptoms ['healthy'] => []
Participant 71 Symptoms ['headache', 'weakness'] => ['headache', 'weakness']
Participant 2036 Symptoms ['weakness'] => ['weakness']
Participant 2518 Symptoms ['healthy'] => []
Participant 2038 Symptoms ['abdominal_pain', 'weakness'] => ['abdominal_pain', 'weakness']
Participant 1641 Symptoms ['healthy'] => []
Participant 2842 Symptoms ['healthy'] => []
Participant 2665 Symptoms ['diarrhea'] => ['diarrhea']
Partic

In [24]:
for part_to_sbv_dict in part_to_sbv_dicts:
    print(len(part_to_sbv_dict.keys()))

216
343
1327


In [25]:
severe_symptoms = ['chest_pain', 'dyspnea', 'heat over 38.9', 'confusion', 'chills']

In [26]:
# Get Healthy and Sick Participant Numbers

def create_healthy_sick_participant_arrays(vaccine_num):
    participant_num_to_symptoms = dict()
    days_since_vaccine = vaccines_to_analyze[vaccine_num]
    vaccine_label = vaccine_labels[vaccine_num]
    symptoms_df = symptoms_dfs[vaccine_num]
    part_to_symptoms_before_vaccine = part_to_sbv_dicts[vaccine_num]

    severe_participants = list()
    mild_participants = list()
    healthy_participants = list()
    all_participants = list()
    missing_day_before_vaccine_participants = list()

    df = symptoms_df.where(symptoms_df[days_since_vaccine]>=0).where(symptoms_df[days_since_vaccine]<=2).dropna(how='all')

    symptoms_list = list()
    for i,r in df.iterrows():
        symptoms_old = r["quest_symptoms_processed"].split(',')
        symptoms = preprocess_symptom_array(symptoms_old)

        participant = r["participant_num"]
        
        # If they don't have data from the day before the vaccine, we won't be able to tell
        # if their symptoms are new from the vaccine or not.
        # I exclude healthy people who did not fill out the form the day before the vaccine
        # as to not bias the findings to include those who were healthy.
        try:
            symptoms_before_vaccine = part_to_symptoms_before_vaccine[participant]

            if symptoms == [''] or symptoms == ['skipped_question'] or symptoms == ['-1']:
                # Participants who skipped the questionnaire after the vaccine get skipped.
                pass
            elif 'healthy' in symptoms or symptoms == ['no heat']:
                # Healthy participants just get added to all_participants
                try:
                    participant_num_to_symptoms[participant]
                except:
                    participant_num_to_symptoms[participant] = list()
                    all_participants.append(participant)
            else:
                # Iterate through all of the symptoms and classify the new ones into
                # Severe and Mild reactions.
                for s in symptoms:
                    if s in possible_symptoms and s != '' and s != 'no heat':
                        if s not in symptoms_before_vaccine:
                            if s in severe_symptoms:
                                severe_participants.append(participant)
                            else:
                                mild_participants.append(participant)
                            # Add the new symptom to the list of symptoms
                            try:
                                participant_num_to_symptoms[participant].append(s)
                            except:
                                participant_num_to_symptoms[participant] = list()
                                participant_num_to_symptoms[participant].append(s)
                                all_participants.append(participant)
                        else:
                            # Participant with no new symptoms just get added to all_participants
                            try:
                                participant_num_to_symptoms[participant]
                            except:
                                participant_num_to_symptoms[participant] = list()
                                all_participants.append(participant)
        except:
            missing_day_before_vaccine_participants.append(participant)
            
    
    for k, v in participant_num_to_symptoms.items():
        if len(v) == 0:
            participant_num_to_symptoms[k] = ['healthy'] 
        else:
            participant_num_to_symptoms[k] = list(set(v))
            
                        

    # Generate healthy participants by excluding sick participant from all participants we have data for
    severe_participants = list(set(severe_participants))
    mild_participants = list(set(mild_participants))
    mild_participants = [part for part in mild_participants if part not in severe_participants]
    all_participants = list(set(all_participants))
    print("%s Severe Participant Numbers: %s" % (vaccine_label, severe_participants))
    print("%s Mild Participant Numbers: %s" % (vaccine_label, mild_participants))
    healthy_participants = [part for part in all_participants if part not in severe_participants and part not in mild_participants]
    
    assert(set(severe_participants).isdisjoint(mild_participants))
    assert(set(severe_participants).isdisjoint(healthy_participants))
    assert(set(mild_participants).isdisjoint(healthy_participants))
    assert(set(all_participants).isdisjoint(missing_day_before_vaccine_participants))
    
    print("Severe n=%d\tMild n=%d\tHealthy n=%d\tAll n=%d\tMissing n=%d" % (len(severe_participants),
                                                                            len(mild_participants),
                                                              len(healthy_participants),
                                                              len(all_participants),
                                                              len(missing_day_before_vaccine_participants)))
    
    return severe_participants, mild_participants, healthy_participants, all_participants, missing_day_before_vaccine_participants, participant_num_to_symptoms

In [27]:

#First Vaccination Severe Participant Numbers: ['63', '1108', '137', '198', 'p1_223', 'p1_8', '68', '36']
#First Vaccination Mild Participant Numbers: ['187', 'p1_17', 'p1_255', '169', '53', 'p1_400', '85', '153', '99', '161', '557', 'p1_561', '93', '32', '118', '73', 'p1_154', '41', '48', 'p1_237', '230', 'p1_20', '329', '28', '50', '119', 'p1_606', '136', '49', 'p1_590', 'p1_211']
#Severe n=8	Mild n=31	Healthy n=154	All n=193	Missing n=27

v1severe, v1mild, v1healthy, v1all, v1missing, v1participant_num_to_symptoms = create_healthy_sick_participant_arrays(0)
v2severe, v2mild, v2healthy, v2all, v2missing, v2participant_num_to_symptoms = create_healthy_sick_participant_arrays(1)
v3severe, v3mild, v3healthy, v3all, v3missing, v3participant_num_to_symptoms = create_healthy_sick_participant_arrays(2)

First Vaccination Severe Participant Numbers: ['1108', '137', '198', 'p1_8', '68', '36']
First Vaccination Mild Participant Numbers: ['p1_400', '53', '85', '153', 'p1_539', '230', '187', '169', '32', 'p1_20', 'p1_32', 'p1_17', 'p1_223', '49', '118', '119', '41', '48', '136', 'p1_606', '99', 'p1_154', 'p1_590', '557', '28', '161', 'p1_211', 'p1_237', '329', 'p1_244', '50', 'p1_561', '73', '93']
Severe n=6	Mild n=34	Healthy n=164	All n=204	Missing n=25
Second Vaccination Severe Participant Numbers: ['p1_634', 'p1_642', 'p1_551', 'p1_616', 'p1_204', 'p1_8', 'p1_614', '169', 'p1_3', '32', 'p1_522', 'p1_595', 'p1_584', 'p1_16', 'p1_525', '49', 'p1_536', '24', 'p1_615', 'p1_622', '103', 'p1_600', 'p1_4', '48', 'p1_577', '136', 'p1_509', '18', '99', '54', 'p1_501', 'p1_553', 'p1_629', 'p1_597', 'p1_303', '161', 'p1_403', 'p1_515', '21', 'p1_113', '603', 'p1_618', 'p1_244', 'p1_549', '50', 'p1_592', 'p1_617', 'p1_550', 'p1_547', '163', 'p1_588', 'p1_559']
Second Vaccination Mild Participant Nu

In [28]:
v1_participant_nums = [v1severe, v1mild, v1healthy, v1all, v1missing]
v2_participant_nums = [v2severe, v2mild, v2healthy, v2all, v2missing]
v3_participant_nums = [v3severe, v3mild, v3healthy, v3all, v3missing]

with open('v1_participant_nums.pkl', 'wb') as f:
    pkl.dump(v1_participant_nums, f)
    
with open('v2_participant_nums.pkl', 'wb') as f:
    pkl.dump(v2_participant_nums, f)
    
with open('v3_participant_nums.pkl', 'wb') as f:
    pkl.dump(v3_participant_nums, f)
    
with open('v1_participant_nums_to_symptoms.pkl', 'wb') as f:
    pkl.dump(v1participant_num_to_symptoms, f)
    
with open('v2_participant_nums_to_symptoms.pkl', 'wb') as f:
    pkl.dump(v2participant_num_to_symptoms, f)
    
with open('v3_participant_nums_to_symptoms.pkl', 'wb') as f:
    pkl.dump(v3participant_num_to_symptoms, f)